# Step 2.7: bars and hierarchical skipping in the Rust engine

This notebook is the check of step 2.7. The Rust engine now fills orders on bars, 1-second, 1-minute, and an import's coarser 5-minute or daily bars, by replaying each bar as four pseudo-ticks, as `docs/spec/fill-semantics.md` says (F-BAR-01 to F-BAR-07). It can also skip the records that no order can act on: it reads a 1-minute bar first and descends to 1-second bars and to trades only where the bar's range reaches a working order's price, or an order becomes active inside it. Skipping must change nothing, so every run here is made twice, skipping and in full, and compared.

The notebook runs every scenario of `fixtures/scenarios/` through both engines in both replays and shows two of them. It then generates a year of ES sessions as dense as the committed fixtures, builds its engine cache, and times a strategy over it in both replays. The year is generated by seed into your `FEFAT_HOME` cache folder, about 1.4 GB with its cache, and may be deleted at any time; nothing here reads your store.

## The two engines

The Rust engine is the binary `just build-engine` writes into the repository's `engine/target/release/`; set `FEFAT_CLI` to use another. The reference engine is `fefat.reference.engine`.

Expected on this PC: `The Rust engine: ...\engine\target\release\fefat-cli.exe`.

In [ ]:
import os
import subprocess
import sys
import time
from datetime import date, timedelta
from pathlib import Path

import polars as pl

import fefat
from fefat.config import resolve_paths
from fefat.market import instruments
from fefat.market.engine_cache import build_engine_cache
from fefat.market.fidelity import TickFidelity
from fefat.market.store import Store
from fefat.reference.rust_batch import CacheCase, Replay
from fefat.reference.rust_run import (
    RustEngine,
    compare_engines,
    default_binary,
    engine_cases,
    explained,
)
from fefat.reference.scenarios import load_scenarios
from fefat.reference.settings import RunSettings
from fefat.reference.units import instrument_parameters
from fefat.strategy.spec import StrategySpec

REPOSITORY = Path(fefat.__file__).resolve().parents[4]
engine = RustEngine(default_binary())
print(f"The Rust engine: {engine.binary}")

## Every scenario, in both replays

Every scenario of `fixtures/scenarios/` runs through both engines, those on bars of every timeframe and those on trades and TBBO records. The Rust engine runs them twice, as one batch each time: in full, every record and bar in turn, and skipping, with the coarser bars that a tape's records make above them, so that the descent decides what to replay (a tape of ticks gets 1-second and 1-minute bars, one of 1-second bars 1-minute bars, and one of 1-minute bars 5-minute bars). For each, the two engines must give the same fills, price, time, rule, flags, liquidity, slippage, and cash, the same event log, entry by entry, and the same count of ambiguous fills.

Expected: `62 scenarios: 62 agree with the reference engine in both replays`, then `30 of them on bars`.

In [ ]:
scenarios = load_scenarios(REPOSITORY / "fixtures" / "scenarios")
cache_folder = resolve_paths(os.environ).cache_dir
comparisons = {}
for replay in Replay:
    for comparison in compare_engines(engine, engine_cases(scenarios, cache_folder, replay)):
        comparisons[(comparison.case.scenario.id, replay)] = comparison
differing = {
    scenario_id for (scenario_id, _), comparison in comparisons.items() if comparison.differences()
}
on_bars = [scenario for scenario in scenarios if not isinstance(scenario.fidelity, TickFidelity)]
agreeing = len(scenarios) - len(differing)
print(f"{len(scenarios)} scenarios: {agreeing} agree with the reference engine in both replays")
print(f"{len(on_bars)} of them on bars")
for (scenario_id, replay), comparison in comparisons.items():
    for difference in comparison.differences():
        print(f"{scenario_id}, {replay}: {difference}")

## S-fill-022, a stop and a target inside one bar

A long holds a stop at 4995.00 and a target at 5010.00, one OCO group. The bar from 14:31 to 14:32 reaches both. Holding a long, its path goes to the low first (F-BAR-02), so the stop fills a tick below its price, 4994.75, at the bar's close (F-BAR-06). With its extremes the other way round the target would have filled, so the fill is flagged ambiguous (F-BAR-05). The run shown is the skipping one.

Expected: the expected fill `order 1 at 4994.75 at 2026-03-06T14:32:00Z (F-STP-04)`, and both engines' fill at that price, flagged `ambiguous`.

In [ ]:
print(explained(comparisons[("S-fill-022", Replay.SKIPPING)]))

## S-template-006, a trade at a bar's close

A template's long has its stop at 4999.25. A trade received exactly at the 14:31 signal close, at 4999.00, belongs to the next minute's bar, yet it comes before that close's decision (F-CLK-02): it fills the stop, and the decision, which signals an exit, finds the position flat. A descent that ran a bar's records, then its decision, then the next bar would fill that trade too late. The skipping run descends from 1-minute bars here and must end each block of records before the next decision.

Expected: the fills the scenario expects, then from both engines `order 1 buy 1 at 5000.25 at 2026-03-06T14:30:00.2Z (F-MKT-02, taker, 1 ticks); order 2 sell 1 at 4999 at 2026-03-06T14:31:00Z (F-STP-03, taker, 1 ticks)`.

In [ ]:
print(explained(comparisons[("S-template-006", Replay.SKIPPING)]))

## A generated year

The year is ES's weekday sessions of 2026, as dense as the committed fixtures, about 71,000 trades a session, each session with three planted quiet stretches of a few minutes and three price jumps. `scripts/gen_fixtures.py` generates it by seed into `<FEFAT_HOME>/cache/synthetic-es-2026-01-02-2026-12-31/`; its engine cache holds the trades, the 1-second and 1-minute bars, and the offsets between them (step 2.5). Generating it takes about two minutes the first time; later the cell reuses it, and the build finds every month up to date. This also completes step 2.5's check of building a year of three levels: the build's time and size are printed here.

Expected on this PC, the first time: `generated 260 sessions in 112 s`, then `ES (synthetic) 2026: 12 contract-months built, 0 up to date, 260 trade dates, 957.9 MB in 10.3 s`, give or take a few seconds. Later runs print only the build line, with `0 contract-months built, 12 up to date`.

In [ ]:
first = date(2026, 1, 2)
weekdays = [first + timedelta(days=day) for day in range(364)]
weekdays = [day for day in weekdays if day.weekday() < 5]
sessions = int(os.environ.get("FEFAT_CHECK_DAYS", len(weekdays)))
last = weekdays[sessions - 1]
year = resolve_paths(os.environ).cache_dir / f"synthetic-es-{first}-{last}"
data = year / "data"
if not (data / "synthetic" / "MANIFEST.json").is_file():
    started = time.perf_counter()
    generator = REPOSITORY / "scripts" / "gen_fixtures.py"
    dates = ["--dates", f"{first}..{last}", "--contracts", "ESH2026", "--planted-gaps", "3"]
    command = [sys.executable, str(generator), "--out", str(data / "synthetic"), *dates]
    generated = subprocess.run(command, capture_output=True, text=True, check=False)
    if generated.returncode != 0:
        raise RuntimeError(f"Expected gen_fixtures.py to pass, got: {generated.stderr}")
    print(f"generated {sessions} sessions in {time.perf_counter() - started:.0f} s")
started = time.perf_counter()
build = build_engine_cache(Store(data), year / "cache", "ES", 2026)
built = build.summary().splitlines()[0].removesuffix(".")
print(f"{built} in {time.perf_counter() - started:.1f} s")
contract = year / "cache" / "engine" / "synthetic" / "ES" / "ESH2026"

## Skipping against the full replay

A strategy runs over the year: at each 5-minute close it enters long when the price stands above where it stood an hour before and short when below, with a bracket 16 ticks against it and 32 ticks for it, one position at a time. It runs through the Rust engine on the year's trades, skipping by their 1-minute and 1-second bars, and again in full, three times each, one after the other, each run a process of its own. All six must give the same fills, event log, ledger, and count of ambiguous fills. On trades no fill is ambiguous: only a bar's path between its extremes is a guess (F-BAR-05).

Expected: `identical`, then `15,071 fills, 0 of them ambiguous` on the whole year.

In [ ]:
FIVE_MINUTES = 300 * 1_000_000_000
minutes = pl.scan_parquet((data / "synthetic" / "ohlcv-1m").as_posix() + "/**/*.parquet")
signals = (
    minutes.select("ts_open", "close")
    .with_columns(ts_close=(pl.col("ts_open") // FIVE_MINUTES + 1) * FIVE_MINUTES)
    .sort("ts_open")
    .group_by("ts_close", maintain_order=True)
    .agg(pl.col("close").last())
    .sort("ts_close")
    .with_columns(change=pl.col("close") - pl.col("close").shift(12))
    .select(
        "ts_close",
        long_entry=pl.col("change") > 0,
        short_entry=pl.col("change") < 0,
        long_exit=pl.lit(False),
        short_exit=pl.lit(False),
    )
    .collect()
)
spec = StrategySpec.model_validate(
    {
        "spec_version": 1,
        "instrument": "ES",
        "timeframe": "5m",
        "execution": {
            "entry": {"kind": "market"},
            "stop": {"kind": "ticks", "ticks": 16},
            "targets": [{"price": {"kind": "ticks", "ticks": 32}}],
            "sizing": {"kind": "fixed", "units": 1},
        },
    }
)


def timed(replay):
    case = CacheCase(
        id=f"the generated year, {replay}",
        folder=contract,
        fidelity=TickFidelity.TRADES,
        instrument=instrument_parameters(instruments.get("ES")),
        settings=RunSettings.for_fidelity(TickFidelity.TRADES),
        strategy=spec,
        signals=signals,
        replay=replay,
    )
    started = time.perf_counter()
    (outcome,) = engine.run_on_caches([case])
    return outcome, time.perf_counter() - started


runs = {Replay.SKIPPING: [], Replay.FULL: []}
for _ in range(3):
    for replay in runs:
        runs[replay].append(timed(replay))
outcomes = [outcome for timings in runs.values() for outcome, _ in timings]
print("identical" if all(outcome == outcomes[0] for outcome in outcomes) else "different")
print(f"{len(outcomes[0].fills):,} fills, {outcomes[0].ambiguous_fills} of them ambiguous")

## The time each replay takes

Each run's time is that of its whole process: starting `fefat-cli`, mapping the cache, reading the signal rows, replaying, and writing the outputs. The skipping engine passes over the minutes and seconds in which no order can act, so it must be faster; how much depends on how often the strategy's orders sit within a bar's range.

Expected on this PC: the skipping runs take about 0.45 s each and the full ones about 1.3 s, then `the skipping engine is N times as fast as the full replay`, N about 3 (2.7 to 3.0 in the runs of step 2.7, lower while the PC is busy).

In [ ]:
for replay, timings in runs.items():
    seconds = sorted(elapsed for _, elapsed in timings)
    print(f"{replay}: {seconds[0]:.2f} to {seconds[-1]:.2f} s over {len(seconds)} runs")
fastest = {replay: min(elapsed for _, elapsed in timings) for replay, timings in runs.items()}
ratio = fastest[Replay.FULL] / fastest[Replay.SKIPPING]
print(f"the skipping engine is {ratio:.1f} times as fast as the full replay")

## Your checks

1. The comparison cell prints `identical`.
2. The timing cell shows the skipping engine faster than the full replay.